In [1]:
import ratinabox 
from ratinabox.Environment import Environment 
from ratinabox.Agent import Agent
from ratinabox.Neurons import Neurons, PlaceCells, RandomSpatialNeurons, GridCells
from ratinabox.contribs.NeuralNetworkNeurons import NeuralNetworkNeurons

from tqdm import tqdm
from PIL import Image

if ratinabox.USE_CUPY:
    import cupy as np
    import cupyx.scipy as scipy
    from cupyx.scipy import stats as stats
else:
    import numpy as np
    import scipy
    from scipy import stats as stats


import matplotlib.pyplot as plt

In [2]:
#Globals 
N_FEATURES = 200 # number of input features
HIDDEN_LAYER_SIZES = [100,100,100] # number of neurons in each hidden layer
LR = 2*1e-3 # learning rate
L2 = 1e-5 # L2 regularization
TAU_E = 10 # eligibilty trace timescale for SGD smoothing

In [3]:
Env = Environment() # 1mm for high resolution plotting 
Ag = Agent(Env,params={'dt':0.1})

In [4]:
from cupyx.scipy.interpolate import RegularGridInterpolator

In [5]:

class ImageNeurons(Neurons):
    """Recieves any .png image, normalises it, and scales it to fill the environment. The firing rate of the neuron is then the value of the image at the Agents current position."""
    default_params = {"image_path":None}
    def __init__(self,Ag,params={}):
        params['n']=1
        super().__init__(Ag,params)
        self.image = np.array(Image.open(self.image_path))[:,:,0]/255  # load image and normalise
        extent = self.Agent.Environment.extent
        self.X = np.linspace(extent[0],extent[1],self.image.shape[1])
        self.Y = np.linspace(extent[2],extent[3],self.image.shape[0])[::-1]
        self.image_interpolated = RegularGridInterpolator(points=(self.Y,self.X),values=self.image) # create interpolator
    def get_state(self,evaluate_at="agent", **kwargs):
        if evaluate_at == "agent":
            pos = self.Agent.pos
        elif evaluate_at == "all":
            pos = self.Agent.Environment.flattened_discrete_coords
        else:
            pos = kwargs["pos"]
        pos = np.array(pos).reshape(-1, pos.shape[-1])
        firingrate = self.image_interpolated((pos[:,1],pos[:,0])).reshape(self.n,-1) # evaluate image at position (or array of positions)
        return firingrate
    


In [6]:
Target = ImageNeurons(Ag,params={"image_path":"../../.images/demos/riab_target.png"})


In [7]:
import torch 
import torch.nn as nn

#Generic neural network class
class MultiLayerPerceptron(nn.Module):
    def __init__(self, n_in=100, n_out=1, n_hidden=[100,100]):
        nn.Module.__init__(self)
        n = [n_in] + n_hidden + [n_out]
        layers = nn.ModuleList()
        for i in range(len(n)-1):
            layers.append(nn.Linear(n[i],n[i+1]))
            if i < len(n)-2: layers.append(nn.ReLU()) #add a ReLU after each hidden layer (but not the last)
        self.net = nn.Sequential(*layers)
    def forward(self, X):
        return self.net(X)

In [8]:
neural_network_module = MultiLayerPerceptron(n_in=N_FEATURES, n_hidden=HIDDEN_LAYER_SIZES, n_out=1).cuda()

neural_network_module

MultiLayerPerceptron(
  (net): Sequential(
    (0): Linear(in_features=200, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=100, bias=True)
    (3): ReLU()
    (4): Linear(in_features=100, out_features=100, bias=True)
    (5): ReLU()
    (6): Linear(in_features=100, out_features=1, bias=True)
  )
)

In [9]:
def loss_function(output, target):
    """Takes the output and input and returns the loss. This function will work whether you pass these in as numpy or torch arrays but, of course, loss will only be backpropagatable if the output is a torch tensor attached to the computational graph."""
    #Make sure they're both torch tensors 
    #Calculate the mean squared error 
    loss = torch.mean((output - target)**2)
    return loss


# Lists to save loss into and a function to plot it
Ag.history['loss'] = [] #<-- we'll store loss data over training here
def plot_loss_history(losses):
    """Plots the loss history. Losses is a list of [[time,loss],...]"""
    losses = np.array(losses)
    fig, ax = plt.subplots(figsize=(4,2))
    ax.plot(losses[:,0]/60,losses[:,1],alpha=1,c='C2')
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Loss")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0,max(losses[:,1])+0.01)
    return fig, ax

In [10]:
optimiser = torch.optim.SGD(neural_network_module.parameters(), lr=LR*Ag.dt**2, momentum=(1-(Ag.dt/TAU_E)), weight_decay=L2,nesterov=True)

In [11]:
Inputs = GridCells(Ag,params={
                            'n':N_FEATURES,
                            'gridscale':np.linspace(0.2,0.5,N_FEATURES)})

NNN = NeuralNetworkNeurons(Ag, params={'input_layers':[Inputs], #<-- the inputs to the network 
                                       'NeuralNetworkModule':neural_network_module, #<-- the neural network which maps these inputs to outputs 
                                       })

# Visualise 
# fig, ax = Inputs.plot_rate_map(chosen_neurons='5')
# fig.suptitle("The inputs to the network")
# fig, ax = NNN.plot_rate_map()
# fig.suptitle("Neural network before training")

In [13]:
TRAIN_MINS = 300
try: 
    for i in (pbar := tqdm(range(int((TRAIN_MINS*60)/Ag.dt)))):
        #update neurons
        Ag.update()
        Inputs.update()
        NNN.update()
        Target.update()

        #zero gradients
        NNN.NeuralNetworkModule.zero_grad()

        #backpropagate
        loss = loss_function(output=NNN.firingrate_torch,
                             target=torch.as_tensor(Target.firingrate, device='cuda'))
        loss.backward()
        optimiser.step()
        
        #Every minute esitmate loss over entire environment
        if i % int(60/Ag.dt) == 0:
            full_loss = loss_function(output = torch.as_tensor(NNN.get_state(evaluate_at='all'), device='cuda'),
                                  target = torch.as_tensor(Target.get_state(evaluate_at='all'), device='cuda')
                                  )
            Ag.history['loss'].append([Ag.t,full_loss.item()])


        pbar.set_description(f"Time (min): {int(Ag.t/60)}, Loss: {full_loss:.4f}")
except KeyboardInterrupt: #allow user to interupt the simulation
    pass


Time (min): 300, Loss: 0.0150: 100%|██████████| 180000/180000 [25:48<00:00, 116.26it/s]
